In [15]:
import requests
import polars as pl 

LAT, LON = 33.74, -118.27

url = "https://archive-api.open-meteo.com/v1/archive"
params = {
    "latitude": LAT,
    "longitude": LON,
    "start_date": "2023-06-01",
    "end_date": "2025-12-31",
    "hourly": ["wind_speed_10m", "wind_gusts_10m", "precipitation", "weather_code"],
    "timezone": "America/Los_Angeles"
}

response = requests.get(url, params=params)
weather = pl.DataFrame(response.json()["hourly"])
weather = weather.with_columns(
    pl.col("time").str.to_datetime().alias("hour_key")
)

print(weather.head())
print(f"Rows: {len(weather)}")

shape: (5, 6)
┌────────────────┬────────────────┬────────────────┬───────────────┬──────────────┬────────────────┐
│ time           ┆ wind_speed_10m ┆ wind_gusts_10m ┆ precipitation ┆ weather_code ┆ hour_key       │
│ ---            ┆ ---            ┆ ---            ┆ ---           ┆ ---          ┆ ---            │
│ str            ┆ f64            ┆ f64            ┆ f64           ┆ i64          ┆ datetime[μs]   │
╞════════════════╪════════════════╪════════════════╪═══════════════╪══════════════╪════════════════╡
│ 2023-06-01T00: ┆ 6.3            ┆ 17.3           ┆ 0.0           ┆ 3            ┆ 2023-06-01     │
│ 00             ┆                ┆                ┆               ┆              ┆ 00:00:00       │
│ 2023-06-01T01: ┆ 4.2            ┆ 14.0           ┆ 0.0           ┆ 3            ┆ 2023-06-01     │
│ 00             ┆                ┆                ┆               ┆              ┆ 01:00:00       │
│ 2023-06-01T02: ┆ 2.4            ┆ 9.7            ┆ 0.0           ┆ 3       

In [16]:
marine_url = "https://marine-api.open-meteo.com/v1/marine"
marine_params = {
    "latitude": LAT,
    "longitude": LON,
    "start_date": "2023-06-01",
    "end_date": "2025-12-31",
    "hourly": [
        "wave_height",
        "swell_wave_height",
    ],
    "timezone": "America/Los_Angeles"
}

response = requests.get(marine_url,params = marine_params)
waves = pl.DataFrame(response.json()["hourly"])
waves = waves.with_columns(pl.col("time").str.to_datetime().alias("hour_key"))
print(waves.head())
print(f"Rows: {len(waves)}")

shape: (5, 4)
┌──────────────────┬─────────────┬───────────────────┬─────────────────────┐
│ time             ┆ wave_height ┆ swell_wave_height ┆ hour_key            │
│ ---              ┆ ---         ┆ ---               ┆ ---                 │
│ str              ┆ f64         ┆ f64               ┆ datetime[μs]        │
╞══════════════════╪═════════════╪═══════════════════╪═════════════════════╡
│ 2023-06-01T00:00 ┆ 0.76        ┆ 0.46              ┆ 2023-06-01 00:00:00 │
│ 2023-06-01T01:00 ┆ 0.76        ┆ 0.5               ┆ 2023-06-01 01:00:00 │
│ 2023-06-01T02:00 ┆ 0.76        ┆ 0.54              ┆ 2023-06-01 02:00:00 │
│ 2023-06-01T03:00 ┆ 0.76        ┆ 0.56              ┆ 2023-06-01 03:00:00 │
│ 2023-06-01T04:00 ┆ 0.78        ┆ 0.56              ┆ 2023-06-01 04:00:00 │
└──────────────────┴─────────────┴───────────────────┴─────────────────────┘
Rows: 22680


In [20]:
# Weather timezone fix
weather_utc = weather.with_columns(
    pl.col("hour_key").dt.replace_time_zone(
        "America/Los_Angeles", ambiguous="latest", non_existent="null"
    ).dt.convert_time_zone("UTC")
     .dt.replace_time_zone(None)
     .alias("hour_key")
).drop("time").drop_nulls("hour_key")

waves_utc = waves.with_columns(
    pl.col("hour_key").dt.replace_time_zone(
        "America/Los_Angeles", ambiguous="latest", non_existent="null"
    ).dt.convert_time_zone("UTC")
     .dt.replace_time_zone(None)
     .alias("hour_key")
).drop("time").drop_nulls("hour_key")




# Merge weather + waves
weather_all = weather_utc.join(waves_utc, on="hour_key", how="left")
print(f"Weather features: {weather_all.shape}")

# Load AIS
ais = pl.concat([
    pl.read_parquet("data/ais_2023H2.parquet"),
    pl.read_parquet("data/ais_2024H1.parquet"),
    pl.read_parquet("data/ais_2024H2.parquet"),
    pl.read_parquet("data/ais_2025H1.parquet"),
    pl.read_parquet("data/ais_2025H2.parquet"),
])


print(f"AIS: {ais.shape}")

# Join
ais = ais.with_columns(
    pl.col("hour_key").dt.replace_time_zone(None)
)
df = ais.join(weather_all, on="hour_key", how="left")
print(f"Merged: {df.shape}")
print(f"Weather nulls: {df['wind_speed_10m'].null_count()}")

# Save
df.write_parquet("data/processed/ais_2023_2025.parquet")
print(f"Saved: {df.shape}")


Weather features: (22678, 7)
AIS: (21678617, 48)
Merged: (21678617, 54)
Weather nulls: 8160
Saved: (21678617, 54)
